In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

conf = pyspark.SparkConf().setAll([
    ('spark.master', 'local[6]'),  # use all 6 cores
    ('spark.app.name', 'Basic Setup'),
    ('spark.driver.memory', '6g'),  # leave some RAM for the OS
    ('spark.executor.memory', '6g'),
    ('spark.sql.shuffle.partitions', '12')  # 2x cores
])

spark = SparkSession.builder.config(conf=conf).getOrCreate()

schema = StructType([
    StructField("subject_id", IntegerType(), True),
    StructField("activity_label", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("x", FloatType(), True),
    StructField("y", FloatType(), True),
    StructField("z", StringType(), True),
])

In [2]:
watch_accelDF = spark.read.csv("hdfs:///watch_accel/*.txt", schema=schema)
watch_gyroDF = spark.read.csv("hdfs:///watch_gyro/*.txt", schema=schema)
phone_accelDF = spark.read.csv("hdfs:///phone_accel/*.txt", schema=schema)
phone_gyroDF = spark.read.csv("hdfs:///phone_gyro/*.txt", schema=schema)

In [3]:
watch_accelDF.show(5)

+----------+--------------+----------------+----------+----------+-----------+
|subject_id|activity_label|       timestamp|         x|         y|          z|
+----------+--------------+----------------+----------+----------+-----------+
|      1638|             A|1138138097322000|  7.302415|-5.4199295|  4.485872;|
|      1638|             A|1138138117418000| 6.5407987|-3.3218923|0.71371585;|
|      1638|             A|1138138137546000| 3.2644117|-2.7231374|0.22513184;|
|      1638|             A|1138138157791000| 1.0705738| -3.319497| 1.3771362;|
|      1638|             A|1138138177887000|-1.6214283|-3.8272414| 1.0035132;|
+----------+--------------+----------------+----------+----------+-----------+
only showing top 5 rows


In [4]:
watch_accelDF = watch_accelDF.withColumn("z", regexp_replace(col("z").cast("string"), ";", "").cast("float"))
watch_gyroDF = watch_gyroDF.withColumn("z", regexp_replace(col("z").cast("string"), ";", "").cast("float"))
phone_accelDF = phone_accelDF.withColumn("z", regexp_replace(col("z").cast("string"), ";", "").cast("float"))
phone_gyroDF = phone_gyroDF.withColumn("z", regexp_replace(col("z").cast("string"), ";", "").cast("float"))

In [5]:
watch_accel = watch_accelDF.select(
    "subject_id", "activity_label", "timestamp",
    col("x").alias("wa_x"), col("y").alias("wa_y"), col("z").alias("wa_z")
)
watch_gyro = watch_gyroDF.select(
    "subject_id", "timestamp",
    col("x").alias("wg_x"), col("y").alias("wg_y"), col("z").alias("wg_z")
)
phone_accel = phone_accelDF.select(
    "subject_id", "activity_label", "timestamp",
    col("x").alias("pa_x"), col("y").alias("pa_y"), col("z").alias("pa_z")
)
phone_gyro = phone_gyroDF.select(
    "subject_id", "timestamp",
    col("x").alias("pg_x"), col("y").alias("pg_y"), col("z").alias("pg_z")
)

In [6]:
watch_accel.show(10, truncate=False)

+----------+--------------+----------------+----------+----------+----------+
|subject_id|activity_label|timestamp       |wa_x      |wa_y      |wa_z      |
+----------+--------------+----------------+----------+----------+----------+
|1638      |A             |1138138097322000|7.302415  |-5.4199295|4.485872  |
|1638      |A             |1138138117418000|6.5407987 |-3.3218923|0.71371585|
|1638      |A             |1138138137546000|3.2644117 |-2.7231374|0.22513184|
|1638      |A             |1138138157791000|1.0705738 |-3.319497 |1.3771362 |
|1638      |A             |1138138177887000|-1.6214283|-3.8272414|1.0035132 |
|1638      |A             |1138138198015000|-5.760022 |-4.974456 |0.07185059|
|1638      |A             |1138138218251000|-9.101074 |-7.592212 |0.15088624|
|1638      |A             |1138138238347000|-3.65959  |-10.40157 |2.799778  |
|1638      |A             |1138138258507000|5.5684204 |-10.085427|2.907554  |
|1638      |A             |1138138278682000|8.58854   |-7.161108

In [7]:
window_schema = StructType([
    StructField("subject_id",     IntegerType(), False),
    StructField("activity_label", StringType(),  False),
    StructField("window_index",   IntegerType(), False),
    StructField("features",       ArrayType(ArrayType(FloatType())), False),
])

In [8]:
MAX_ROWS_PER_ACTIVITY = 3600  # 3 min × 20Hz — truncate anything beyond this
TARGET_HZ        = 20
WINDOW_SECONDS   = 10
STEP_SECONDS     = 5                              # 50% overlap
SAMPLES_PER_WIN  = TARGET_HZ * WINDOW_SECONDS    # 200
STEP_SAMPLES     = TARGET_HZ * STEP_SECONDS      # 100

def extract_windows(pdf):
    pdf = pdf.sort_values("timestamp").reset_index(drop=True)

    pdf["z"] = pdf["z"].astype(str).str.replace(";", "", regex=False).astype(float)

    subject_id     = int(pdf["subject_id"].iloc[0])
    activity_label = str(pdf["activity_label"].iloc[0])

    signals = pdf[["x", "y", "z"]].values.astype(np.float32)[:MAX_ROWS_PER_ACTIVITY]

    rows = []
    window_index = 0

    for start in range(0, len(signals) - SAMPLES_PER_WIN + 1, STEP_SAMPLES):
        window = signals[start : start + SAMPLES_PER_WIN]
        if len(window) < SAMPLES_PER_WIN:
            break
        rows.append({
            "subject_id":     subject_id,
            "activity_label": activity_label,
            "window_index":   window_index,
            "features":       window.tolist(),
        })
        window_index += 1

    if not rows:
        return pd.DataFrame(columns=["subject_id", "activity_label", "window_index", "features"])

    return pd.DataFrame(rows)

In [9]:
wdw_watch_a = (
    watch_accelDF
    .groupBy("subject_id", "activity_label")
    .applyInPandas(extract_windows, schema=window_schema)
)

wdw_watch_g = (
    watch_gyroDF
    .groupBy("subject_id", "activity_label")
    .applyInPandas(extract_windows, schema=window_schema)
)

wdw_phone_a = (
    phone_accelDF
    .groupBy("subject_id", "activity_label")
    .applyInPandas(extract_windows, schema=window_schema)
)

wdw_phone_g = (
    phone_gyroDF
    .groupBy("subject_id", "activity_label")
    .applyInPandas(extract_windows, schema=window_schema)
)

In [10]:
wdw_watch_a.show(1)

[Stage 8:>                                                          (0 + 1) / 1]

+----------+--------------+------------+--------------------+
|subject_id|activity_label|window_index|            features|
+----------+--------------+------------+--------------------+
|      1600|             D|           0|[[-0.1666963, 1.5...|
+----------+--------------+------------+--------------------+
only showing top 1 row


Traceback (most recent call last):
  File "/usr/spark-4.1.1/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/spark-4.1.1/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe
                                                                                

In [16]:
base = "file:///home/work/activity_classifier/windowed"
wdw_watch_a.write.parquet(f"{base}/watch_accel/", mode="overwrite")
wdw_watch_g.write.parquet(f"{base}/watch_gyro/", mode="overwrite")
wdw_phone_a.write.parquet(f"{base}/phone_accel/", mode="overwrite")
wdw_phone_g.write.parquet(f"{base}/phone_gyro/", mode="overwrite")